In [ ]:
import importlib.util
import sys

IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
print(IN_COLAB, sys.executable)

In [ ]:
from pathlib import Path

MOUNT_DRIVE = True

cache = Path("../../../cache/data/modelnet40_1024.npz")
if IN_COLAB:
    cache = Path("/content/data/modelnet40_1024.npz")
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = Path("/content/drive/MyDrive/Documents/SHAD/modelnet40_1024.npz")
    cache.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
import sys
import tarfile
from pathlib import Path

import requests
from tqdm import tqdm

url = "https://huggingface.co/datasets/Pointcept/modelnet40_normal_resampled-compressed/resolve/main/modelnet40_normal_resampled.tar.gz"
data_dir = Path("/content/data") if IN_COLAB else Path("../../../cache/data")
archive = data_dir / "modelnet40_normal_resampled.tar.gz"
modelnet_dir = data_dir / "modelnet40_normal_resampled"
data_dir.mkdir(exist_ok=True)

if cache.exists():
    print(f"Кэш уже есть, архив не нужен: {cache}")
elif len(list(modelnet_dir.glob("*/*.txt"))) == 12311:
    print(f"Данные уже распакованы: {modelnet_dir.resolve()}")
else:
    if archive.exists():
        print(f"Архив уже скачан: {archive.resolve()}")
    else:
        print(f"Скачиваю {url}")
        part = archive.with_suffix(".part")
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            total = int(r.headers["content-length"])
            with open(part, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="download", file=sys.stdout) as bar:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    bar.update(len(chunk))
        part.rename(archive)

    print(f"Распаковываю в {modelnet_dir.resolve()}")
    modelnet_dir.mkdir(exist_ok=True)
    with open(archive, "rb") as raw, tqdm.wrapattr(raw, "read", total=archive.stat().st_size, desc="extract", file=sys.stdout) as f:
        with tarfile.open(fileobj=f, mode="r|gz") as tar:
            tar.extractall(modelnet_dir, filter="data")
    print("classes:", len([p for p in modelnet_dir.iterdir() if p.is_dir()]))
    print("shapes:", len(list(modelnet_dir.glob("*/*.txt"))))

In [ ]:
import importlib.util
import sys
from pathlib import Path

if IN_COLAB and not Path("points").exists():
    !git clone https://github.com/AlisherBlack/points.git
if IN_COLAB and not Path("Pointnet_Pointnet2_pytorch").exists():
    !git clone --depth 1 -b master-only-scripts https://github.com/AlisherBlack/Pointnet_Pointnet2_pytorch.git
if IN_COLAB and not Path("dgcnn").exists():
    !git clone --depth 1 https://github.com/WangYueFt/dgcnn.git

for path in ["points", "Pointnet_Pointnet2_pytorch/models"]:
    path = str(Path(path).resolve())
    if path not in sys.path:
        sys.path.insert(0, path)

viz = importlib.import_module("points.viz")
importlib.reload(viz)

In [ ]:
from points.modelnet import modelnet40_splits, prepare

cache = prepare(modelnet_dir, num_points=1024, cache=cache) # читаем первые N точек, не случайные
train_set, test_set = modelnet40_splits(cache, normals=False)
classes = test_set.classes
print(len(train_set), len(test_set))

In [ ]:
viz.show_modelnet(cache, per_class=4, num_points=1024, height=780)

## PointNet

- До агрегации — общий (shared) MLP из 5 слоёв: $3 \to 64 \to 64 \to 64 \to 128 \to 1024$. Каждый слой — `Conv1d` 1×1 + BN + ReLU, применяется к каждой точке независимо. 
- Max pool по точкам → вектор 1024
- После агрегации — голова из 3 линейных слоёв: $1024 \to 512 \to 256$ (Linear + BN + ReLU + Dropout 0.3) и финальный `Linear(256, 40)` → логиты классов (`num_classes`)
- 1-в-1 как в статье PointNet, но без T-Net

In [ ]:
import torch
from torch import nn


def conv_bn_relu(in_channels, out_channels):
    return nn.Sequential(nn.Conv1d(in_channels, out_channels, kernel_size=1), nn.BatchNorm1d(out_channels), nn.ReLU())


def linear_bn_relu_dropout(in_features, out_features, dropout):
    return nn.Sequential(nn.Linear(in_features, out_features), nn.BatchNorm1d(out_features), nn.ReLU(), nn.Dropout(dropout))


class PointNet(nn.Module):
    def __init__(self, num_classes, in_channels, widths, head, dropout):
        super().__init__()
        channels = (in_channels, *widths)
        self.point_mlp = nn.Sequential(*[conv_bn_relu(a, b) for a, b in zip(channels[:-1], channels[1:])])
        features = (widths[-1], *head)
        self.head = nn.Sequential(
            *[linear_bn_relu_dropout(a, b, dropout) for a, b in zip(features[:-1], features[1:])],
            nn.Linear(head[-1], num_classes),
        )

    def forward(self, points):
        features = self.point_mlp(points.transpose(1, 2))
        pooled = features.max(dim=2).values
        return self.head(pooled)

    def critical_points(self, points):
        return self.point_mlp(points.transpose(1, 2)).argmax(dim=2)

In [ ]:
from points.train import count_parameters, fit, forward_ms, get_device, plot_losses

TRAIN_MODELS = False
EPOCHS = 15
BATCH_SIZE = 32

device = get_device()
print("device:", device)

torch.manual_seed(0)
pointnet = PointNet(
    num_classes=40,
    in_channels=3,
    widths=(64, 64, 64, 128, 1024),
    head=(512, 256),
    dropout=0.3,
).to(device).eval()
print(f"параметров: {count_parameters(pointnet) / 1e6:.1f}M")

points, _ = test_set[0]
points = points.unsqueeze(0).to(device)
permutation = torch.randperm(points.shape[1], device=device)
with torch.no_grad():
    difference = (pointnet(points) - pointnet(points[:, permutation])).abs().max()
print("изменение выхода при перестановке точек:", float(difference))

batch = torch.stack([test_set[i][0] for i in range(BATCH_SIZE)])
pointnet_forward_ms = forward_ms(pointnet, batch, device)
print(f"forward: {pointnet_forward_ms:.1f} мс (батч {BATCH_SIZE})")

In [ ]:
from IPython.display import Markdown, display

results = {}


def report(name, model, ms, result=None):
    accuracy = f"{result['test_accuracy']:.3f}" if result else "—"
    results[name] = f"| {name} | {count_parameters(model) / 1e6:.2f} | {ms:.1f} | {accuracy} |"
    header = [f"| модель | параметры, M | forward, мс (батч {BATCH_SIZE}) | test accuracy |", "|---|---|---|---|"]
    display(Markdown("\n".join(header + list(results.values()))))

In [ ]:
train_set[0][0].shape

In [ ]:
pointnet_result = None
if TRAIN_MODELS:
    pointnet_result = fit(pointnet, train_set, test_set, device, epochs=EPOCHS, batch_size=BATCH_SIZE)
    plot_losses(pointnet_result["history"])
report("PointNet", pointnet, pointnet_forward_ms, pointnet_result)

In [ ]:
from points.viz import show_critical_points

sample, label = test_set[0]
pointnet.eval()
with torch.no_grad():
    pred = pointnet(sample[None].to(device)).argmax().item()
    critical = pointnet.critical_points(sample[None].to(device))[0].unique().cpu()
title = f"{classes[label]} → {classes[pred]} · критических точек {len(critical)} из {len(sample)}"
show_critical_points(sample, critical, title)

## PointNet++

In [ ]:
GRID = 0.2
STAGES = 3

torch.manual_seed(0)
t = torch.rand(300) * 2 * torch.pi
ring = 3 * torch.stack([t.cos(), t.sin()], dim=1) + 0.1 * torch.randn(300, 2)
blob = 0.4 * torch.randn(300, 2) + torch.tensor([0.8, 0.4])
coord = torch.cat([ring, blob])
coord -= coord.min(0).values
feat = torch.randn(len(coord), 8)

EXTENT = float(torch.ceil(coord.max() / (GRID * 2**STAGES)) * GRID * 2**STAGES)
coord.shape, feat.shape

In [ ]:
from plotly.subplots import make_subplots


def show(panels, cols=2, gone=None):
    rows = (len(panels) + cols - 1) // cols
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=[title for title, *_ in panels], horizontal_spacing=0.05, vertical_spacing=0.06)
    for k, (_, points, size, cell) in enumerate(panels):
        row, col = k // cols + 1, k % cols + 1
        if gone is not None:
            fig.add_scatter(x=gone[k][:, 0].numpy(), y=gone[k][:, 1].numpy(), mode="markers", marker=dict(color="#c8c8c8", size=4), row=row, col=col)
        fig.add_scatter(x=points[:, 0].numpy(), y=points[:, 1].numpy(), mode="markers", marker=dict(color="#1f77b4", size=size), row=row, col=col)
        fig.update_xaxes(dtick=cell, row=row, col=col)
        fig.update_yaxes(dtick=cell, row=row, col=col)
    axis = dict(range=[0, EXTENT], constrain="domain", tick0=0, gridcolor="#9a9a9a", showline=True, linecolor="#9a9a9a", mirror=True, showticklabels=False, zeroline=False)
    fig.update_xaxes(**axis, matches="x")
    fig.update_yaxes(**axis, matches="y", scaleanchor="x")
    fig.update_layout(width=1000, height=520 * rows, showlegend=False, plot_bgcolor="white", margin=dict(l=10, r=10, t=40, b=10))
    fig.show()

### Случайный выбор vs FPS

K центров, вокруг каждого — шар радиуса R (ball query), как в PointNet++

In [ ]:
K_MAX = 48
R = 0.6


def fps(coord, k):
    idx = torch.zeros(k, dtype=torch.long)
    dist = torch.full((len(coord),), float("inf"))
    for i in range(1, k):
        dist = torch.minimum(dist, (coord - coord[idx[i - 1]]).norm(dim=1))
        idx[i] = dist.argmax()
    return idx


def uncovered(centers):
    return int((torch.cdist(coord, centers) > R).all(dim=1).sum())


def show_balls(panels):
    fig = make_subplots(rows=1, cols=len(panels), subplot_titles=[f"{name}: K = 0, вне шаров {len(coord)}" for name, _ in panels], horizontal_spacing=0.05)
    for col, (_, centers) in enumerate(panels, start=1):
        fig.add_scatter(x=coord[:, 0].numpy(), y=coord[:, 1].numpy(), mode="markers", marker=dict(color="#1f77b4", size=4), row=1, col=col)
        fig.add_scatter(x=[], y=[], mode="markers", marker=dict(color="#d62728", size=8), row=1, col=col)
        for x, y in centers.tolist():
            fig.add_shape(type="circle", x0=x - R, y0=y - R, x1=x + R, y1=y + R, line=dict(color="#d62728", width=1), visible=False, row=1, col=col)

    steps = []
    for k in range(K_MAX + 1):
        data = {"x": [centers[:k, 0].tolist() for _, centers in panels], "y": [centers[:k, 1].tolist() for _, centers in panels]}
        layout = {f"annotations[{i}].text": f"{name}: K = {k}, вне шаров {uncovered(centers[:k])}" for i, (name, centers) in enumerate(panels)}
        layout |= {f"shapes[{i}].visible": i % K_MAX < k for i in range(len(panels) * K_MAX)}
        steps.append(dict(method="update", label=str(k), args=[data, layout, [1, 3]]))

    axis = dict(range=[0, EXTENT], constrain="domain", showgrid=False, showline=True, linecolor="#9a9a9a", mirror=True, showticklabels=False, zeroline=False)
    fig.update_xaxes(**axis, matches="x")
    fig.update_yaxes(**axis, matches="y", scaleanchor="x")
    fig.update_layout(
        width=1000, height=600, showlegend=False, plot_bgcolor="white", margin=dict(l=10, r=10, t=40, b=10),
        sliders=[dict(active=0, steps=steps, currentvalue=dict(visible=False), pad=dict(t=20), font=dict(size=9))],
    )
    fig.show()


random_centers = coord[torch.randperm(len(coord))[:K_MAX]]
fps_centers = coord[fps(coord, K_MAX)]
show_balls([("случайно", random_centers), ("FPS", fps_centers)])

### Оригинальный PointNet++

Оригинальный PointNet++ (SSG), сами не реализовываем, а берем из [Pointnet_Pointnet2_pytorch](https://github.com/yanx27/Pointnet_Pointnet2_pytorch) не обучаем — только замеряем forward. Обёртка `PointNet2Orig`, чтобы подогнать под нужный формат: вход $(B, N, 3) \to (B, 3, N)$, из выхода берётся `log_softmax`

In [ ]:
from torch import nn

from pointnet2_cls_ssg import get_model


class PointNet2Orig(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = get_model(num_classes, normal_channel=False)

    def forward(self, points):
        return self.model(points.transpose(1, 2))[0]


torch.manual_seed(0)
pointnet2_orig = PointNet2Orig(num_classes=40).to(device).eval()
print(f"параметров: {count_parameters(pointnet2_orig) / 1e6:.1f}M")

with torch.no_grad():
    difference = (pointnet2_orig(points) - pointnet2_orig(points[:, permutation])).abs().max()
print("изменение выхода при перестановке точек:", float(difference))

pointnet2_orig_forward_ms = forward_ms(pointnet2_orig, batch, device)
print(f"forward: {pointnet2_orig_forward_ms:.1f} мс (батч {BATCH_SIZE})")
report("Оригинальный PointNet++", pointnet2_orig, pointnet2_orig_forward_ms)

### GridSample

train: из каждой ячейки — одна случайная точка. eval: берем тот, кт ближе к центру. Идея взята из PTv2

In [ ]:
def with_batch(grid, batch):
    return grid if batch is None else torch.cat([batch[:, None], grid], dim=1)


def split_batch(key, batch):
    return (key, None) if batch is None else (key[:, 1:], key[:, 0])


def grid_sample(coord, grid_size, batch=None, training=True):
    grid = torch.floor(coord / grid_size).long()
    voxels, inverse = torch.unique(with_batch(grid, batch), dim=0, return_inverse=True)
    score = torch.rand(len(coord), device=coord.device) if training else (coord / grid_size - grid - 0.5).norm(dim=1)
    best = score.new_full((len(voxels),), float("inf")).scatter_reduce_(0, inverse, score, "amin")
    winner = score == best[inverse]
    index = torch.arange(len(coord), device=coord.device)
    pick = index.new_full((len(voxels),), len(coord)).scatter_reduce_(0, inverse[winner], index[winner], "amin")
    grid, batch = split_batch(voxels, batch)
    return grid, pick, batch


grid_coord, pick, _ = grid_sample(coord, GRID)
sampled, sampled_feat = coord[pick], feat[pick]

show([
    (f"исходные: N = {len(coord)}", coord, 5, GRID),
    (f"GridSample {GRID}: N = {len(sampled)}", sampled, 5, GRID),
])

### Grid pooling

Ячейка ×2: `grid_coord >> 1`, `coord` — среднее, `feat` — max

In [ ]:
def scatter(x, index, size, reduce):
    return x.new_zeros(size, x.shape[1]).scatter_reduce_(0, index[:, None].expand_as(x), x, reduce, include_self=False)


def grid_cluster(grid, batch=None, shift=1):
    parents, cluster = torch.unique(with_batch(grid >> shift, batch), dim=0, return_inverse=True)
    parents, batch = split_batch(parents, batch)
    return parents, cluster, batch


def grid_pool(grid, coord, feat, batch=None, shift=1):
    parents, cluster, batch = grid_cluster(grid, batch, shift)
    coord = scatter(coord, cluster, len(parents), "mean")
    feat = scatter(feat, cluster, len(parents), "amax")
    return parents, coord, feat, batch


g, c, f = grid_coord, sampled, sampled_feat
stages = [(g, c)]
for _ in range(STAGES):
    g, c, f, _ = grid_pool(g, c, f)
    stages.append((g, c))

panels = []
for stage, (g, c) in enumerate(stages):
    cell = GRID * 2**stage
    name = "grid sampling" if stage == 0 else "grid pooling"
    panels.append((f"{name} {cell:g}: N = {len(g)}", c, 6, cell))
show(panels, gone=[coord] + [c for _, c in stages[:-1]])

#### ModelNet

Ячейку увеличиваем в grid pooling, а не в grid sampling:

- grid sampling оставляет одну случайную точку на ячейку, а остальные выбрасывает до того, как их увидит сеть;
- grid pooling агрегирует все точки, попавшие в ячейку (в модели — MLP по каждой, потом max): информация не теряется, а сжимается.

grid sampling 0.05 → grid pooling `>> 2` (0.2) → `>> 1` (0.4). Серые — точки предыдущего уровня

In [ ]:
MN_GRID = 0.05
SHIFTS = (2, 1)

sample, _ = test_set[0]
g, pick, _ = grid_sample(sample, MN_GRID)
c = sample[pick]
panels = [
    (f"сэмпл: N = {len(sample)}", sample, None, 2.5),
    (f"grid sampling {MN_GRID:g}: N = {len(g)}", c, sample, 2.5),
]
cell = MN_GRID
for stage, shift in enumerate(SHIFTS, start=1):
    gone = c
    g, c, _, _ = grid_pool(g, c, c, shift=shift)
    cell *= 2**shift
    panels.append((f"grid pooling {cell:g}: N = {len(g)}", c, gone, 2.5 + 1.5 * stage))

fig = make_subplots(rows=2, cols=2, specs=[[{"type": "scene"}] * 2] * 2, subplot_titles=[title for title, *_ in panels], horizontal_spacing=0.02, vertical_spacing=0.05)
for k, (_, xyz, gone, size) in enumerate(panels):
    row, col = k // 2 + 1, k % 2 + 1
    if gone is not None:
        fig.add_scatter3d(
            x=gone[:, 0].numpy(), y=gone[:, 1].numpy(), z=gone[:, 2].numpy(), mode="markers",
            marker=dict(color="#c8c8c8", size=2.5), row=row, col=col,
        )
    fig.add_scatter3d(
        x=xyz[:, 0].numpy(), y=xyz[:, 1].numpy(), z=xyz[:, 2].numpy(), mode="markers",
        marker=dict(color="#1f77b4", size=size), row=row, col=col,
    )
fig.update_scenes(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False, camera=dict(up=dict(x=0, y=1, z=0), eye=dict(x=1.4, y=0.9, z=1.4)))
fig.update_layout(width=1000, height=900, showlegend=False, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

### PointNet++ на сетке

Чтобы не ждать миллион лет, сделаем упрощение. Возьмем сетку (grid pooling), как предлагают в PTv2 и используем его вместо ball query и fps. Ширины MLP и голова — как в официальном коде PointNet++, кроме выходов SA1 и SA2: 144 и 288 вместо 128 и 256, чтобы ниже делить их на головы attention по 18 каналов (задел под APE и PointROPE)

- Grid sampling 0.05 внутри `forward`: в train — случайная точка вокселя, в eval — ближайшая к центру вокселя
- SA1: grid pooling `>> 2` (ячейка 0.2) → общий MLP по $x_j - c$: $3 \to 64 \to 64 \to 144$ → max по ячейке
- SA2: grid pooling `>> 1` (ячейка 0.4) → общий MLP по $[x_j - c,\ f_j]$: $147 \to 128 \to 128 \to 288$ → max по ячейке
- SA3 (`group_all`): общий MLP по $[c,\ f]$: $291 \to 256 \to 512 \to 1024$ → max по объекту
- Голова: $1024 \to 512 \to 256$ (Linear + BN + ReLU + Dropout 0.3) и `Linear(256, 40)`
- $c$ — среднее координат точек ячейки, слой MLP — Linear + BN + ReLU

SA — Set Abstraction, термин из PN2

In [ ]:
def linear_bn_relu(in_features, out_features):
    return nn.Sequential(nn.Linear(in_features, out_features), nn.BatchNorm1d(out_features), nn.ReLU())


def shared_mlp(in_channels, widths):
    channels = (in_channels, *widths)
    return nn.Sequential(*[linear_bn_relu(a, b) for a, b in zip(channels[:-1], channels[1:])])


class GridSetAbstraction(nn.Module):
    def __init__(self, in_channels, widths, shift):
        super().__init__()
        self.shift = shift
        self.mlp = shared_mlp(in_channels + 3, widths) # 3 канала под отно координаты

    def forward(self, grid, coord, feat, batch):
        parents, cluster, batch = grid_cluster(grid, batch, self.shift)
        center = scatter(coord, cluster, len(parents), "mean")
        x = [coord - center[cluster]]
        if feat is not None:
            x.append(feat)
        x = self.mlp(torch.cat(x, dim=1))
        return parents, center, scatter(x, cluster, len(parents), "amax"), batch


class PointNet2Grid(nn.Module):
    def __init__(self, num_classes, grid_size, shifts, widths, global_widths, head, dropout):
        super().__init__()
        self.grid_size = grid_size
        channels = (0, *[w[-1] for w in widths])
        self.sa = nn.ModuleList([GridSetAbstraction(c, w, s) for c, w, s in zip(channels[:-1], widths, shifts)])
        self.global_mlp = shared_mlp(channels[-1] + 3, global_widths)
        features = (global_widths[-1], *head)
        self.head = nn.Sequential(
            *[linear_bn_relu_dropout(a, b, dropout) for a, b in zip(features[:-1], features[1:])],
            nn.Linear(head[-1], num_classes),
        )

    def forward(self, points):
        B, n, _ = points.shape
        coord = points.reshape(-1, 3)
        batch = torch.arange(B, device=points.device).repeat_interleave(n)
        grid, pick, batch = grid_sample(coord, self.grid_size, batch, self.training)
        coord = coord[pick]
        feat = None
        for stage in range(len(self.sa)):
            grid, coord, feat, batch = self.stage(stage, grid, coord, feat, batch, B)
        x = self.global_mlp(torch.cat([coord, feat], dim=1))
        return self.head(scatter(x, batch, B, "amax"))

    def stage(self, i, grid, coord, feat, batch, num_objects):
        return self.sa[i](grid, coord, feat, batch)

In [ ]:
torch.manual_seed(0)
pointnet2 = PointNet2Grid(
    num_classes=40,
    grid_size=0.05,
    shifts=(2, 1), # сколько бит сдвигаем
    widths=((64, 64, 144), (128, 128, 288)), # SA1, SA2
    global_widths=(256, 512, 1024), # global MLP
    head=(512, 256),
    dropout=0.3,
).to(device).eval()
print(f"параметров: {count_parameters(pointnet2) / 1e6:.2f}M")

with torch.no_grad():
    difference = (pointnet2(points) - pointnet2(points[:, permutation])).abs().max()
print("изменение выхода при перестановке точек:", float(difference))

pointnet2_forward_ms = forward_ms(pointnet2, batch, device)
print(f"forward: {pointnet2_forward_ms:.1f} мс (батч {BATCH_SIZE})")

In [ ]:
pointnet2_result = None
if TRAIN_MODELS:
    pointnet2_result = fit(pointnet2, train_set, test_set, device, epochs=EPOCHS, batch_size=BATCH_SIZE)
    plot_losses(pointnet2_result["history"])
report("PointNet++ на сетке", pointnet2, pointnet2_forward_ms, pointnet2_result)

## DGCNN

Оригинальный DGCNN, сами не будем реализовывать (долгий форвард), а берем из [dgcnn](https://github.com/WangYueFt/dgcnn). Обёртка `DGCNNOrig`, чтобы подогнать под нужный формат: вход $(B, N, 3) \to (B, 3, N)$

In [ ]:
import types
from pathlib import Path

from torch import nn

source = Path("dgcnn/pytorch/model.py").read_text()
# В `get_graph_feature` у авторов зашито `torch.device('cuda')`
# при загрузке меняем на устройство входа, остальной код без изменений.
patched = source.replace("device = torch.device('cuda')", "device = x.device")
assert patched != source

dgcnn_repo = types.ModuleType("dgcnn_repo")
exec(compile(patched, "dgcnn/pytorch/model.py", "exec"), dgcnn_repo.__dict__)


class DGCNNOrig(nn.Module):
    def __init__(self, num_classes, k, emb_dims, dropout):
        super().__init__()
        args = types.SimpleNamespace(k=k, emb_dims=emb_dims, dropout=dropout)
        self.model = dgcnn_repo.DGCNN(args, output_channels=num_classes)

    def forward(self, points):
        return self.model(points.transpose(1, 2))


torch.manual_seed(0)
dgcnn = DGCNNOrig(
    num_classes=40,
    k=20,
    emb_dims=1024,
    dropout=0.5,
).to(device).eval()
print(f"параметров: {count_parameters(dgcnn) / 1e6:.1f}M")

with torch.no_grad():
    difference = (dgcnn(points) - dgcnn(points[:, permutation])).abs().max()
print("изменение выхода при перестановке точек:", float(difference))

dgcnn_forward_ms = forward_ms(dgcnn, batch, device)
print(f"forward: {dgcnn_forward_ms:.1f} мс (батч {BATCH_SIZE})")
report("Оригинальный DGCNN", dgcnn, dgcnn_forward_ms)

Оригинальный DGCNN не обучаем для чистоты эксперимента (+ долго). Кроме EdgeConv он отличается от PointNet ещё и след

- skip-concat выходов всех слоёв ($64 + 64 + 128 + 256 = 512$);
- max + avg pooling вместо одного max;
- LeakyReLU вместо ReLU;
- dropout 0.5 вместо 0.3.

Поэтому влияние EdgeConv проверяем на PointNet + EdgeConv.

In [ ]:
def edge_conv(x, k, h):                                              
    # x: (B, n, F)
    # cross-distance
    idx = torch.cdist(x, x, p=2).topk(k, largest=False).indices      # (B, n, k)
    # в каком батче искать соседа
    b = torch.arange(x.shape[0], device=x.device)[:, None, None]     # (B, 1, 1)
    xj = x[b, idx]                                                   # (B, n, k, F)
    xi = x[:, :, None].expand_as(xj)                                 # (B, n, k, F)
    e = h(torch.cat([xi, xj - xi], dim=-1))                          # (B, n, k, F')
    return e.max(dim=2).values                                       # (B, n, F')


class EdgeConv(nn.Module):
    def __init__(self, in_channels, out_channels, k):
        super().__init__()
        self.k = k
        self.linear = nn.Linear(in_channels * 2, out_channels)
        self.bn = nn.BatchNorm1d(out_channels)

    def h(self, edges):
        e = self.linear(edges)
        e = self.bn(e.reshape(-1, e.shape[-1])).reshape(e.shape)
        return torch.relu(e)

    def forward(self, x):                                            # x: (B, F, n)
        x = x.transpose(1, 2)                                        # (B, n, F)
        x = edge_conv(x, self.k, self.h)                             # (B, n, F')
        return x.transpose(1, 2)                                     # (B, F', n)


class PointNetEdge(PointNet):
    def __init__(self, in_channels, widths, edge_layers, k, **kwargs):
        super().__init__(in_channels=in_channels, widths=widths, **kwargs)
        channels = (in_channels, *widths)
        for i in range(edge_layers):
            self.point_mlp[i] = EdgeConv(channels[i], channels[i + 1], k)

In [ ]:
torch.manual_seed(0)
pointnet_edge = PointNetEdge(
    num_classes=40,
    in_channels=3,
    widths=(64, 64, 64, 128, 1024),
    head=(512, 256),
    dropout=0.3,
    edge_layers=2,
    k=20,
).to(device).eval()
print(f"параметров: {count_parameters(pointnet_edge) / 1e6:.1f}M")

with torch.no_grad():
    difference = (pointnet_edge(points) - pointnet_edge(points[:, permutation])).abs().max()
print("изменение выхода при перестановке точек:", float(difference))

pointnet_edge_forward_ms = forward_ms(pointnet_edge, batch, device)
print(f"forward: {pointnet_edge_forward_ms:.1f} мс (батч {BATCH_SIZE})")

In [ ]:
pointnet_edge_result = None
if TRAIN_MODELS:
    pointnet_edge_result = fit(pointnet_edge, train_set, test_set, device, epochs=EPOCHS, batch_size=BATCH_SIZE)
    plot_losses(pointnet_edge_result["history"])
report("PointNet + EdgeConv", pointnet_edge, pointnet_edge_forward_ms, pointnet_edge_result)

## PointNet++ на сетке + attention

### Сериализация

Код ячейки сетки — чередование битов её координат. Сортировка по коду задаёт порядок обхода ячеек: так PTv3 вытягивает облако в последовательность

#### Чередование битов

2D, сетка 8×8 — 3 бита на ось. Биты $x$ и $y$ кладём по очереди, в паре старший — $x$.

In [ ]:
DEPTH = 3
X, Y = 5, 3

x_bits = f"{X:0{DEPTH}b}"
y_bits = f"{Y:0{DEPTH}b}"
code = int("".join(a + b for a, b in zip(x_bits, y_bits)), 2)

print(f"x    = {'   '.join(x_bits)}")
print(f"y    =   {'   '.join(y_bits)}")
print(f"code = {' '.join(f'{code:0{2 * DEPTH}b}')} = {code}")

То же что и выше, только на тензорах.

In [ ]:
def interleave(axes, depth):
    code = torch.zeros_like(axes[0])
    for bit in range(depth):
        for i, axis in enumerate(axes):
            code |= ((axis >> bit) & 1) << (len(axes) * bit + len(axes) - 1 - i)
    return code


interleave([torch.tensor([X]), torch.tensor([Y])], DEPTH)

#### Порядок обхода

В ячейке — её код, линия — обход по возрастанию кода. z-trans — тот же алгоритм, но с переставленными осями

In [ ]:
SIZE = 2**DEPTH
gx, gy = torch.meshgrid(torch.arange(SIZE), torch.arange(SIZE), indexing="ij")
gx, gy = gx.flatten(), gy.flatten()
code = interleave([gx, gy], DEPTH)
code_trans = interleave([gy, gx], DEPTH)


def show_grid(panels):
    fig = make_subplots(rows=1, cols=len(panels), subplot_titles=[title for title, *_ in panels], horizontal_spacing=0.05)
    for col, (_, code, labels, level) in enumerate(panels, start=1):
        block = 2**level
        for x0 in range(0, SIZE, block):
            for y0 in range(0, SIZE, block):
                fill = "#dbe9f6" if level > 0 and (x0 // block + y0 // block) % 2 else "white"
                border = dict(color="#444444", width=2) if level > 0 else dict(width=0)
                fig.add_shape(type="rect", x0=x0, y0=y0, x1=x0 + block, y1=y0 + block, fillcolor=fill, line=border, layer="below", row=1, col=col)
        path = torch.argsort(code)
        fig.add_scatter(x=(gx[path] + 0.5).numpy(), y=(gy[path] + 0.5).numpy(), mode="lines", line=dict(color="#1f77b4", width=2), row=1, col=col)
        for x, y, label in zip(gx.tolist(), gy.tolist(), labels):
            fig.add_annotation(x=x + 0.5, y=y + 0.5, text=label, showarrow=False, bgcolor="rgba(255, 255, 255, 0.85)", font=dict(size=10, color="#222222"), row=1, col=col)
    axis = dict(range=[0, SIZE], tick0=0, dtick=1, constrain="domain", gridcolor="#c8c8c8", showline=True, linecolor="#9a9a9a", mirror=True, showticklabels=False, zeroline=False)
    fig.update_xaxes(**axis, matches="x")
    fig.update_yaxes(**axis, matches="y", scaleanchor="x")
    fig.update_layout(width=1000, height=520, showlegend=False, plot_bgcolor="white", margin=dict(l=10, r=10, t=40, b=10))
    fig.show()


show_grid([
    ("z: в паре битов старший — x", code, [str(c) for c in code.tolist()], 0),
    ("z-trans: в паре битов старший — y", code_trans, [str(c) for c in code_trans.tolist()], 0),
])

In [ ]:
cube = torch.cartesian_prod(torch.arange(4), torch.arange(4), torch.arange(4))
path = cube[torch.argsort(interleave(cube.unbind(1), 2))]
step = torch.arange(len(path))

fig = make_subplots(rows=1, cols=1, specs=[[{"type": "scene"}]], subplot_titles=["z-order 4×4×4: от светлого к тёмному"])
fig.add_scatter3d(
    x=path[:, 0].numpy(), y=path[:, 1].numpy(), z=path[:, 2].numpy(), mode="lines+markers",
    line=dict(color=step.numpy(), colorscale="Blues", cmin=-20, cmax=len(path) - 1, width=5),
    marker=dict(color=step.numpy(), colorscale="Blues", cmin=-20, cmax=len(path) - 1, size=4),
)
axis = dict(title="", showticklabels=False, tickvals=[0, 1, 2, 3], gridcolor="#c8c8c8", zeroline=False, backgroundcolor="white")
fig.update_scenes(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube", camera=dict(eye=dict(x=1.6, y=-1.6, z=1.1)))
fig.update_layout(width=700, height=650, showlegend=False, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

#### z-order и Гильберт

Кольцо и пятно из раздела PointNet++, уровни grid pooling. Порядок обхода — `argsort(code)`

In [ ]:
def z_order_encode(grid_coord, depth):
    return interleave(grid_coord.long().unbind(1), depth)


def hilbert_encode(grid_coord, depth):
    x = list(grid_coord.long().unbind(1))
    q = 1 << (depth - 1)
    while q > 1:
        p = q - 1
        for i in range(len(x)):
            on = (x[i] & q) > 0
            t = (x[0] ^ x[i]) & p
            x[0] = torch.where(on, x[0] ^ p, x[0] ^ t)
            x[i] = torch.where(on, x[i], x[i] ^ t)
        q >>= 1
    for i in range(1, len(x)):
        x[i] = x[i] ^ x[i - 1]
    t = torch.zeros_like(x[0])
    q = 1 << (depth - 1)
    while q > 1:
        t = torch.where((x[-1] & q) > 0, t ^ (q - 1), t)
        q >>= 1
    return interleave([axis ^ t for axis in x], depth)


def show_order(panels, points, cell):
    fig = make_subplots(rows=1, cols=len(panels), subplot_titles=[title for title, _ in panels], horizontal_spacing=0.05)
    for col, (_, order) in enumerate(panels, start=1):
        path = points[order]
        fig.add_scatter(
            x=path[:, 0].numpy(), y=path[:, 1].numpy(), mode="lines+markers",
            line=dict(color="#444444", width=1), marker=dict(color=torch.arange(len(path)).numpy(), colorscale="Viridis", size=6),
            row=1, col=col,
        )
    axis = dict(range=[0, EXTENT], constrain="domain", tick0=0, dtick=cell, gridcolor="#9a9a9a", showline=True, linecolor="#9a9a9a", mirror=True, showticklabels=False, zeroline=False)
    fig.update_xaxes(**axis, matches="x")
    fig.update_yaxes(**axis, matches="y", scaleanchor="x")
    fig.update_layout(width=1000, height=520, showlegend=False, plot_bgcolor="white", margin=dict(l=10, r=10, t=40, b=10))
    fig.show()


depth = int(grid_coord.max() + 1).bit_length()
for stage, (g, c) in enumerate(stages[:-1]):
    cell = GRID * 2**stage
    show_order([
        (f"z-order: N = {len(g)}, ячейка {cell:g}", torch.argsort(z_order_encode(g, depth - stage))),
        (f"Гильберт: N = {len(g)}, ячейка {cell:g}", torch.argsort(hilbert_encode(g, depth - stage))),
    ], c, cell)

#### Префикс кода — ячейка после пулинга


- grid pooling `>> 1` — группировка по ячейке после пулинга (в нашем коде через координаты: `unique(grid >> 1)`) плюс агрегация: координата среднее, признак — max.
- `code >> 2` — код ячейки после пулинга, то есть `interleave(x >> 1, y >> 1)`;
- `x >> 1, y >> 1` — координаты ячейки после пулинга;

In [ ]:
for level in (1, 2):
    pooled = interleave([gx >> level, gy >> level], DEPTH - level)
    print(f"code >> {2 * level} == interleave([x >> {level}, y >> {level}]):", torch.equal(code >> 2 * level, pooled))


def binary(code, level):
    return [f"<b>{c >> 2 * level:0{2 * (DEPTH - level)}b}</b>{c & (4**level - 1):0{2 * level}b}" for c in code.tolist()]


show_grid([
    ("code >> 2: блоки 2×2", code, binary(code, 1), 1),
    ("code >> 4: блоки 4×4", code, binary(code, 2), 2),
])

3D: `code >> 3` — код ячейки после пулинга `grid_coord >> 1`: так PTv3 в `SerializedPooling` получает коды ячеек после пулинга без повторного кодирования

In [ ]:
torch.manual_seed(0)
cells = torch.randint(0, 64, (1000, 3))
code_3d = interleave(cells.unbind(1), 6)
for level in (1, 2, 3):
    pooled = interleave((cells >> level).unbind(1), 6 - level)
    print(f"code >> {3 * level} == interleave(cells >> {level}):", torch.equal(code_3d >> 3 * level, pooled))

### Патчи

Patch attention берёт подряд по `PATCH` точек вдоль кривой: патч точки на позиции `i` — `i // PATCH`, как `reshape(-1, PATCH)` в `PatchAttention`. Цвета патчей чередуются, серый пунктир — переход к следующему патчу

In [ ]:
import math

PATCH = 8
PATCH_COLORS = ["#1f77b4", "#ff7f0e", "#d62728"]


def show_patches(panels, points, cell, patch):
    fig = make_subplots(rows=1, cols=len(panels), subplot_titles=[title for title, _ in panels], horizontal_spacing=0.05)
    for col, (_, order) in enumerate(panels, start=1):
        path = points[order]
        fig.add_scatter(
            x=path[:, 0].numpy(), y=path[:, 1].numpy(), mode="lines",
            line=dict(color="#b0b0b0", width=1, dash="dot"), hoverinfo="skip", row=1, col=col,
        )
        for k, chunk in enumerate(path.split(patch)):
            color = PATCH_COLORS[k % len(PATCH_COLORS)]
            fig.add_scatter(
                x=chunk[:, 0].numpy(), y=chunk[:, 1].numpy(), mode="lines+markers", name=f"патч {k}",
                line=dict(color=color, width=2), marker=dict(color=color, size=8), row=1, col=col,
            )
    axis = dict(range=[0, EXTENT], constrain="domain", tick0=0, dtick=cell, gridcolor="#9a9a9a", showline=True, linecolor="#9a9a9a", mirror=True, showticklabels=False, zeroline=False)
    fig.update_xaxes(**axis, matches="x")
    fig.update_yaxes(**axis, matches="y", scaleanchor="x")
    fig.update_layout(width=1000, height=520, showlegend=False, plot_bgcolor="white", margin=dict(l=10, r=10, t=40, b=10))
    fig.show()


for stage, (g, c) in enumerate(stages[:-1]):
    cell = GRID * 2**stage
    title = f"N = {len(g)}, патчей {math.ceil(len(g) / PATCH)}, ячейка {cell:g}"
    show_patches([
        (f"z-order: {title}", torch.argsort(z_order_encode(g, depth - stage))),
        (f"Гильберт: {title}", torch.argsort(hilbert_encode(g, depth - stage))),
    ], c, cell, PATCH)

### Маска патчей

Батч из двух элементов: кольцо — элемент батча 0, пятно — элемент батча 1, по одной точке на ячейку `GRID`. В `PatchAttention` точки всех элементов батча склеены и отсортированы по коду, номер элемента батча — в старших битах: сначала все точки элемента батча 0, потом элемента батча 1, в конце добивка до кратного `PATCH` (pad, `batch = -1`). Патч на стыке захватывает точки двух элементов батча, маска `batch_i == batch_j` запрещает attention между ними

In [ ]:
objects = torch.cat([torch.zeros(len(ring)), torch.ones(len(blob))]).long()

voxels = torch.unique(torch.cat([objects[:, None], torch.floor(coord / GRID).long()], dim=1), dim=0)
objects = voxels[:, 0]
objects.bincount()

Сверху — вся последовательность после сортировки, рамки — окна из трёх патчей. Снизу для каждого окна матрица «кто на кого смотрит»: строка — точка-запрос, столбец — точка-ключ

In [ ]:
import torch.nn.functional as F

MASK_STATES = ["разные патчи: attention не считается", "один патч, маска запрещает", "attention"]
MASK_COLORS = ["#f2f2f2", "#d62728", "#1f77b4"]
OBJECT_NAMES = {0: "элемент батча 0", 1: "элемент батча 1", -1: "pad"}
OBJECT_COLORS = ["#4d4d4d", "#a6a6a6", "#ffffff"]

pad = -len(objects) % PATCH
seq_batch = F.pad(objects, (0, pad), value=-1)
patch = torch.arange(len(seq_batch)) // PATCH
mask = seq_batch[:, None] == seq_batch[None, :]
state = (patch[:, None] == patch[None, :]).long() * (1 + mask.long())

count0 = int((seq_batch == 0).sum())
junction, last = count0 // PATCH, len(seq_batch) // PATCH - 1
windows = [
    (f"стык, патч {junction}:<br>{count0 % PATCH} точек элемента батча 0 + {PATCH - count0 % PATCH} элемента батча 1", (junction - 1) * PATCH),
    (f"конец, патч {last}:<br>{PATCH - pad} точек элемента батча 1 + {pad} pad", (last - 2) * PATCH),
]


def discrete(colors):
    return [[k / len(colors), color] for i, color in enumerate(colors) for k in (i, i + 1)]


def describe(i):
    return f"позиция {i} (патч {i // PATCH}, {OBJECT_NAMES[int(seq_batch[i])]})"


fig = make_subplots(
    rows=2, cols=2, specs=[[{"colspan": 2}, None], [{}, {}]], row_heights=[0.08, 0.92], vertical_spacing=0.1, horizontal_spacing=0.12,
    subplot_titles=[f"вся последовательность: {len(objects)} точек + {pad} pad = {len(seq_batch)} позиций, {last + 1} патчей по {PATCH}"] + [title for title, _ in windows],
)
fig.add_heatmap(
    z=[torch.where(seq_batch < 0, 2, seq_batch).tolist()], zmin=-0.5, zmax=2.5, colorscale=discrete(OBJECT_COLORS), showscale=False,
    text=[[describe(i) for i in range(len(seq_batch))]], hovertemplate="%{text}<extra></extra>", row=1, col=1,
)
for k in range(1, last + 1):
    fig.add_vline(x=k * PATCH - 0.5, line=dict(color="#ffffff", width=1), row=1, col=1)
for b, color in ((0, "#ffffff"), (1, "#222222")):
    fig.add_annotation(x=float((seq_batch == b).nonzero().float().mean()), y=0, text=OBJECT_NAMES[b], showarrow=False, font=dict(color=color), row=1, col=1)
for _, start in windows:
    fig.add_shape(type="rect", x0=start - 0.5, x1=start + 3 * PATCH - 0.5, y0=-0.5, y1=0.5, line=dict(color="#111111", width=2), row=1, col=1)
fig.update_xaxes(range=[-0.5, len(seq_batch) - 0.5], tickfont=dict(size=9), row=1, col=1)
fig.update_yaxes(visible=False, row=1, col=1)

for col, (_, start) in enumerate(windows, start=1):
    idx = torch.arange(start, start + 3 * PATCH)
    z = state[idx][:, idx]
    fig.add_heatmap(
        z=z.numpy(), x=idx.numpy(), y=idx.numpy(), zmin=-0.5, zmax=2.5, colorscale=discrete(MASK_COLORS), showscale=False, xgap=1, ygap=1,
        text=[[f"{describe(int(a))} →<br>{describe(int(b))}<br>{MASK_STATES[v]}" for b, v in zip(idx, row)] for a, row in zip(idx, z.tolist())],
        hovertemplate="%{text}<extra></extra>", row=2, col=col,
    )
    lo, hi = start - 0.5, start + 3 * PATCH - 0.5
    for k in range(1, 3):
        edge = start + k * PATCH - 0.5
        fig.add_shape(type="line", x0=edge, x1=edge, y0=lo, y1=hi, line=dict(color="#222222", width=2), row=2, col=col)
        fig.add_shape(type="line", x0=lo, x1=hi, y0=edge, y1=edge, line=dict(color="#222222", width=2), row=2, col=col)
    for k in range(3):
        fig.add_annotation(
            x=start + k * PATCH + (PATCH - 1) / 2, y=hi, yanchor="top", yshift=-18,
            text=f"патч {start // PATCH + k}", showarrow=False, row=2, col=col,
        )
    fig.update_xaxes(tickvals=idx.numpy(), ticktext=[str(int(i) % PATCH) for i in idx], tickangle=0, tickfont=dict(size=9), constrain="domain", row=2, col=col)
    fig.update_yaxes(
        tickvals=idx.numpy(), ticktext=[OBJECT_NAMES[int(seq_batch[i])] for i in idx], tickfont=dict(size=9),
        autorange="reversed", scaleanchor=f"x{col + 1}", constrain="domain", row=2, col=col,
    )
for name, color in zip(MASK_STATES, MASK_COLORS):
    fig.add_scatter(x=[None], y=[None], mode="markers", marker=dict(symbol="square", size=12, color=color, line=dict(color="#9a9a9a", width=1)), name=name)
fig.update_layout(width=1000, height=720, plot_bgcolor="white", legend=dict(orientation="h", y=-0.1), margin=dict(l=10, r=10, t=40, b=10))
fig.show()

### Архитектура

После SA1 и SA2 — по 2 блока как в PTv3, но без xCPE

- Блок: LN → attention → residual; LN → MLP $C \to 4C \to C$ (GELU) → residual
- Attention по патчам из 8 точек вдоль кривой сериализации, маска не даёт смешивать разные объекты
- Сериализация заново на каждой стадии по `grid_coord` стадии: 4 порядка (z, z-trans, hilbert, hilbert-trans), блок $i$ берёт порядок $i \bmod 4$ (shift order), в train порядки перемешиваются (shuffle)
- Головы: $8 \times 18$ (голова x каналы) после SA1 и $16 \times 18$ после SA2
- Позиционное кодирование: без PE или APE — sin/cos от `grid_coord` стадии, прибавляется к признакам перед блоками
- Последние слои веток (`proj` в attention и второй `Linear` в MLP) инициализированы нулями: без PE модель на старте совпадает с базовой
- lr блоков 1e-4 — в 10 раз меньше остальных весов, как в PTv3 и LitePT

In [ ]:
import torch.nn.functional as F

ORDERS = ("z", "z-trans", "hilbert", "hilbert-trans")


def encode(grid_coord, batch=None, depth=16, order="z"):
    assert order in ORDERS
    if order.endswith("-trans"):
        grid_coord = grid_coord[:, [1, 0, 2]]
    code = z_order_encode(grid_coord, depth) if order.startswith("z") else hilbert_encode(grid_coord, depth)
    if batch is not None:
        code = batch.long() << depth * 3 | code
    return code


def serialize(grid_coord, batch=None, orders=ORDERS, depth=None, shuffle_orders=False):
    """
    Одним проходом по всему батчу
    order — индекс вдоль кривой → индекс точки
    invers —  индекс точки → индекс вдоль кривой
    """
    if depth is None:
        depth = int(grid_coord.max() + 1).bit_length()
    assert depth <= 16
    code = torch.stack([encode(grid_coord, batch, depth, order) for order in orders])
    order = torch.argsort(code)
    inverse = torch.empty_like(order).scatter_(1, order, torch.arange(order.shape[1], device=order.device).expand_as(order))
    if shuffle_orders:
        perm = torch.randperm(len(orders))
        code, order, inverse = code[perm], order[perm], inverse[perm]
    return code, order, inverse, depth


def serialize_stage(grid, batch, num_objects, shuffle_orders):
    grid = grid - scatter(grid.float(), batch, num_objects, "amin").long()[batch]
    _, order, inverse, _ = serialize(grid, batch, shuffle_orders=shuffle_orders)
    return grid, order, inverse


class PatchAttention(nn.Module):
    def __init__(self, channels, num_heads, patch_size):
        super().__init__()
        self.num_heads = num_heads
        self.patch_size = patch_size
        self.qkv = nn.Linear(channels, channels * 3)
        self.proj = nn.Linear(channels, channels)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, feat, batch, order, inverse):
        n, C = feat.shape
        P, H = self.patch_size, self.num_heads
        pad = -n % P # кол-во точек до числа, кратного P
        # batch.shape= (n, )      номер элемента батча для каждой точки
        # order.shape= (n, )
        # inverse.shape= (n, )
        feat = F.pad(feat[order], (0, 0, 0, pad))
        # feat.shape= (n', C)
        batch = F.pad(batch[order], (0, pad), value=-1).reshape(-1, P)
        # batch.shape= (n' / P, P)    номер элемента батча для каждой позиции патча, pad = -1

        
        # q, k, v: (n' / P, H, P, d)      n' = (n_1 + ... + n_B), добитое до кратного P
        # mask:    (n' / P, 1, P, P)      1 растягивается на все H головы
        q, k, v = self.qkv(feat).reshape(-1, P, 3, H, C // H).permute(2, 0, 3, 1, 4)
        mask = (batch[:, :, None] == batch[:, None, :])[:, None]
        x = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        x = x.transpose(1, 2).reshape(-1, C)[:n]
        return self.proj(x)[inverse]


class Block(nn.Module):
    def __init__(self, channels, num_heads, patch_size, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.attn = PatchAttention(channels, num_heads, patch_size)
        self.norm2 = nn.LayerNorm(channels)
        self.mlp = nn.Sequential(nn.Linear(channels, channels * mlp_ratio), nn.GELU(), nn.Linear(channels * mlp_ratio, channels))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, feat, batch, order, inverse):
        feat = feat + self.attn(self.norm1(feat), batch, order, inverse)
        return feat + self.mlp(self.norm2(feat))


class PointNet2GridAttn(PointNet2Grid):
    def __init__(self, widths, attn_depths, num_heads, patch_size, **kwargs):
        super().__init__(widths=widths, **kwargs)
        self.blocks = nn.ModuleList([
            nn.ModuleList([Block(w[-1], h, patch_size) for _ in range(d)])
            for w, h, d in zip(widths, num_heads, attn_depths)
        ])

    def stage(self, i, grid, coord, feat, batch, num_objects):
        grid, coord, feat, batch = super().stage(i, grid, coord, feat, batch, num_objects)
        # pos — коорината ячеек
        pos, order, inverse = serialize_stage(grid, batch, num_objects, self.training)
        feat = self.position(feat, pos)
        for j, block in enumerate(self.blocks[i]):
            # shuffle order
            feat = block(feat, batch, order[j % len(order)], inverse[j % len(order)])
        return grid, coord, feat, batch

    def position(self, feat, pos):
        return feat

In [ ]:
torch.manual_seed(0)
pointnet2_attn = PointNet2GridAttn(
    num_classes=40,
    grid_size=0.05,
    shifts=(2, 1),
    widths=((64, 64, 144), (128, 128, 288)),
    global_widths=(256, 512, 1024),
    head=(512, 256),
    dropout=0.3,
    attn_depths=(2, 2),
    num_heads=(8, 16),
    patch_size=8,
).to(device).eval()
print(f"параметров: {count_parameters(pointnet2_attn) / 1e6:.2f}M")

with torch.no_grad():
    difference = (pointnet2_attn(points) - pointnet2_attn(points[:, permutation])).abs().max()
print("изменение выхода при перестановке точек:", float(difference))

pointnet2_attn_forward_ms = forward_ms(pointnet2_attn, batch, device)
print(f"forward: {pointnet2_attn_forward_ms:.1f} мс (батч {BATCH_SIZE})")

In [ ]:
pointnet2_attn_result = None
if TRAIN_MODELS:
    pointnet2_attn_result = fit(pointnet2_attn, train_set, test_set, device, epochs=EPOCHS, batch_size=BATCH_SIZE, block_lr=1e-4)
    plot_losses(pointnet2_attn_result["history"])
report("PointNet++ на сетке + attention", pointnet2_attn, pointnet2_attn_forward_ms, pointnet2_attn_result)

In [ ]:
def sincos(pos, dim, base=100.0):
    freq = base ** (-torch.arange(0, dim // 3, 2, device=pos.device) / (dim // 3))
    angle = pos[..., None] * freq
    return torch.cat([angle.sin(), angle.cos()], dim=-1).flatten(-2)


class PointNet2GridAPE(PointNet2GridAttn):
    def position(self, feat, pos):
        return feat + sincos(pos.float(), feat.shape[1])

In [ ]:
torch.manual_seed(0)
pointnet2_ape = PointNet2GridAPE(
    num_classes=40,
    grid_size=0.05,
    shifts=(2, 1),
    widths=((64, 64, 144), (128, 128, 288)),
    global_widths=(256, 512, 1024),
    head=(512, 256),
    dropout=0.3,
    attn_depths=(2, 2),
    num_heads=(8, 16),
    patch_size=8,
).to(device).eval()
print(f"параметров: {count_parameters(pointnet2_ape) / 1e6:.2f}M")

with torch.no_grad():
    difference = (pointnet2_ape(points) - pointnet2_ape(points[:, permutation])).abs().max()
print("изменение выхода при перестановке точек:", float(difference))

pointnet2_ape_forward_ms = forward_ms(pointnet2_ape, batch, device)
print(f"forward: {pointnet2_ape_forward_ms:.1f} мс (батч {BATCH_SIZE})")

In [ ]:
pointnet2_ape_result = None
if TRAIN_MODELS:
    pointnet2_ape_result = fit(pointnet2_ape, train_set, test_set, device, epochs=EPOCHS, batch_size=BATCH_SIZE, block_lr=1e-4)
    plot_losses(pointnet2_ape_result["history"])
report("PointNet++ на сетке + attention + APE", pointnet2_ape, pointnet2_ape_forward_ms, pointnet2_ape_result)